In [0]:
import requests
import json

webhook_url = dbutils.secrets.get(
    scope="prj-secret",
    key="slack-webhook"
)

def send_slack_message(message):

    headers = {
        "Content-Type": "application/json"
    }

    payload = {
        "text": message
    }

    requests.post(
        webhook_url,
        headers=headers,
        data=json.dumps(payload)
    )

In [0]:
# Step 1

# Read Bronze table
# Step 1 - Read Bronze Table
from pyspark.sql.functions import *

spark.sql("USE CATALOG Silver_Catalog")
spark.sql("USE SCHEMA Silver_SCH")

device_df = spark.table("Bronze_Catalog.Bronze_SCH.Bronze_Device_Metrics")

In [0]:
# Step 2

# Check Schema

device_df.printSchema()

In [0]:
# Step 4

# Count Records

from pyspark.sql.functions import col
from pyspark.sql.types import DoubleType


device_df = device_df.filter(
    (expr("try_cast(runtime_hours as double)") >= 0) &
    (expr("try_cast(device_power_kw as double)") >= 0) &
    (expr("try_cast(motor_speed_rpm as double)") >= 0) &
    (expr("try_cast(efficiency_ratio as double)") >= 0) &
    (expr("try_cast(energy_draw_kwh as double)") >= 0) &
    (expr("try_cast(heat_output as double)") >= 0) &
    (expr("try_cast(cooling_load as double)") >= 0) &
    (expr("try_cast(device_voltage as double)") >= 0) &
    (expr("try_cast(device_current as double)") >= 0) &
    (expr("try_cast(device_temperature as double)") >= 0)
)

print("Valid Records :", device_df.count())

In [0]:
# Step 5 - Check Null Values

from pyspark.sql.functions import col, sum

null_df = device_df.select(
[
    sum(col(c).isNull().cast("int")).alias(c)
    for c in device_df.columns
]
)

display(null_df)

In [0]:
from pyspark.sql.functions import expr

device_df = device_df \
.withColumn("runtime_hours", expr("try_cast(runtime_hours as double)")) \
.withColumn("device_power_kw", expr("try_cast(device_power_kw as double)")) \
.withColumn("motor_speed_rpm", expr("try_cast(motor_speed_rpm as double)")) \
.withColumn("efficiency_ratio", expr("try_cast(efficiency_ratio as double)")) \
.withColumn("energy_draw_kwh", expr("try_cast(energy_draw_kwh as double)")) \
.withColumn("heat_output", expr("try_cast(heat_output as double)")) \
.withColumn("cooling_load", expr("try_cast(cooling_load as double)")) \
.withColumn("device_voltage", expr("try_cast(device_voltage as double)")) \
.withColumn("device_current", expr("try_cast(device_current as double)")) \
.withColumn("device_temperature", expr("try_cast(device_temperature as double)"))

In [0]:
# Step 7 - Fill Nulls
device_df = device_df.fillna({
    "runtime_hours":0,
    "device_power_kw":0,
    "motor_speed_rpm":0,
    "efficiency_ratio":0,
    "energy_draw_kwh":0,
    "heat_output":0,
    "cooling_load":0,
    "device_voltage":0,
    "device_current":0,
    "device_temperature":0
})

In [0]:
# Step 8 - Verify Schema
device_df.printSchema()

In [0]:
# Step 9 - Data Validation

print("Valid Records :", device_df.count())

In [0]:
# Step 10 - Create Surrogate Key
from pyspark.sql.functions import monotonically_increasing_id

device_df = device_df.withColumn(
    "device_sk",
    monotonically_increasing_id() )

    

In [0]:
# Step 11 - SCD Type 2
from pyspark.sql.functions import current_timestamp, lit

device_df = device_df \
.withColumn("effective_from", current_timestamp()) \
.withColumn("effective_to", lit(None).cast("timestamp")) \
.withColumn("is_current", lit(True))

In [0]:
# Step 12 - Save Silver Table
device_df.write \
.format("delta") \
.mode("overwrite") \
.option("overwriteSchema","true") \
.saveAsTable("Silver_Catalog.Silver_SCH.Silver_Device_Metrics")

In [0]:
from pyspark.sql.functions import current_timestamp

records_loaded = device_df.count()

try:

    spark.sql(f"""
    INSERT INTO Silver_Catalog.Silver_SCH.Audit_Log
    VALUES(
        'Silver_Device_Metrics',
        'Device Metrics Silver Pipeline',
        current_timestamp(),
        {records_loaded},
        'SUCCESS',
        NULL
    )
    """)

    print("✅ Silver_Device_Metrics Loaded Successfully")

    send_slack_message(f"""
✅ Silver_Device_Metrics SUCCESS

Pipeline : Device Metrics Silver Pipeline

Records Loaded : {records_loaded}

Status : SUCCESS
""")

except Exception as e:

    error_message = str(e).replace("'", "''")

    spark.sql(f"""
    INSERT INTO Silver_Catalog.Silver_SCH.Audit_Log
    VALUES(
        'Silver_Device_Metrics',
        'Device Metrics Silver Pipeline',
        current_timestamp(),
        0,
        'FAILED',
        '{error_message}'
    )
    """)

    print(f"❌ Silver_Device_Metrics Failed : {error_message}")

    send_slack_message(f"""
❌ Silver_Device_Metrics FAILED

Pipeline : Device Metrics Silver Pipeline

Error :

{error_message}

Status : FAILURE
""")

    raise

In [0]:
spark.sql("""
OPTIMIZE Silver_Catalog.Silver_SCH.Silver_Device_Metrics
""")

print("✅ OPTIMIZE Completed")

In [0]:
spark.sql("""
OPTIMIZE Silver_Catalog.Silver_SCH.Silver_Device_Metrics
ZORDER BY (household_id)
""")

print("✅ ZORDER Completed")

In [0]:
spark.sql("""
VACUUM Silver_Catalog.Silver_SCH.Silver_Device_Metrics
RETAIN 168 HOURS
""")

print("✅ VACUUM Completed")

In [0]:
spark.sql("""
DESCRIBE HISTORY Silver_Catalog.Silver_SCH.Silver_Device_Metrics
""").show(truncate=False)